# Current DBSCAN model + Ani validation-stage ensemble

Compare a small decision-score ensemble between our existing DBSCAN + one-vs-rest logistic regression model and Ani's validation-stage model from commit `a5487215115e30afb68ab865bc5fd98be6b3e199`. Source notebook snapshot is saved in `ani_source/part1_best_model.ipynb` (SHA256 `06ac000e36d820d9e8c3e96c21b07631e6716e4f29883e726a8fcfc64c1ddbe7`). No final refit with validation labels.

Ani's model is reproduced with its original protocol: scaler/PCA30/diagonal GMM30 use all train/validation/test **features**, without validation/test labels; supervised fitting starts with the 157 verified training trials and self-trains on unlabeled train+test features. Its final validation score must reproduce sample F1 0.893162 and macro F1 0.713700. Our component stays unchanged at 0.910256/0.811421.

The models' scores have different meanings and native cutoffs (ours 0.5; Ani 0.3). Do not call a raw probability average calibrated. Predeclare **threshold-centered log-odds**: `e_ours=logit(p_ours)-logit(0.5)`, `e_ani=logit(p_ani)-logit(0.3)`. Blend `e=(1-alpha)*e_ours+alpha*e_ani`; select scores >=0, with highest-score fallback if empty. Saved `sigmoid(e)` is a decision score, not a calibrated per-label probability. This preserves each component's native predictions at alpha=0/1.

Predeclared Ani weights **0, 0.10, 0.25, 0.50, 1**: three mixed ensembles plus both endpoint controls. No new confidence/cutoff search, validation-label training, or combination with the self-training experiment. Inspect complementary errors, save all scores/models/policies, and prepare a local candidate only if sample F1 beats 0.910256. No Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit,logit
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler,normalize
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
FEATURES=[f"emb_{i}" for i in range(150)]
ANI_WEIGHTS=[0.0,0.10,0.25,0.50,1.0]
ANI_COMMIT='a5487215115e30afb68ab865bc5fd98be6b3e199'
ANI_SNAPSHOT_SHA256='06ac000e36d820d9e8c3e96c21b07631e6716e4f29883e726a8fcfc64c1ddbe7'
EPSILON=1e-12
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT=DATA/"experiments/ani_ensemble"
for folder in ["models","validation_predictions","submissions","datasets"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
assert hashlib.sha256((OUT/"ani_source/part1_best_model.ipynb").read_bytes()).hexdigest()==ANI_SNAPSHOT_SHA256
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
test=pd.read_csv(INPUT_DIR/"part1_test.csv")
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
truth=valid[REASONS].to_numpy(dtype=int);original_y=train[REASONS].to_numpy(dtype=int)
assert len(train)==1171 and known.sum()==157 and len(valid)==78 and len(test)==312
assert train.ID.is_unique and valid.ID.is_unique and test.ID.is_unique
assert set(train.ID).isdisjoint(valid.ID) and set(train.ID).isdisjoint(test.ID) and set(valid.ID).isdisjoint(test.ID)
SOURCE=DATA/"experiments/additional_methods"
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
ours=joblib.load(SOURCE/source_record["model_file"])
protected_paths=[INPUT_DIR/f for f in ["part1_train.csv","part1_val.csv","part1_test.csv"]]+[
    SOURCE/source_record["model_file"],SOURCE/source_record["submission_file"],SOURCE/"datasets/dbscan_m5_q25.csv"]
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Native model cutoffs: ours 0.5, Ani 0.3; supervised validation labels remain held out.")


Native model cutoffs: ours 0.5, Ani 0.3; supervised validation labels remain held out.


In [2]:
def threshold_predictions(scores,cutoff):
    predicted=(scores>=cutoff).astype(int);empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,scores[rows].argmax(axis=1)]=1
    return predicted,empty


def ours_probabilities(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    return np.column_stack([est.predict_proba(scaled)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def fit_ani_base(Xl,Yl):
    rows=[];ys=[];ws=[]
    for i,y in enumerate(Yl):
        positive=np.flatnonzero(y)
        rows.extend([i]*len(positive));ys.extend(positive.tolist());ws.extend([1.0/len(positive)]*len(positive))
    rows=np.asarray(rows);y=np.asarray(ys);weights=np.asarray(ws)
    counts=np.bincount(y,weights=weights,minlength=5)+1e-9
    weights=weights*counts.mean()/counts[y]
    estimator=LogisticRegression(C=0.3,max_iter=3000)
    with warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        estimator.fit(Xl[rows],y,sample_weight=weights)
    return {"lr":estimator,"neighbor_X_normalized":normalize(Xl),"neighbor_targets":Yl/Yl.sum(axis=1,keepdims=True),
            "neighbor_count":10,"knn_weight":0.3}


def predict_ani_base(base,Xq):
    lr_prob=np.zeros((len(Xq),5))
    lr_prob[:,base["lr"].classes_]=base["lr"].predict_proba(Xq)
    similarities=normalize(Xq)@base["neighbor_X_normalized"].T
    indices=np.argsort(-similarities,axis=1)[:,:base["neighbor_count"]]
    weights=np.take_along_axis(similarities,indices,axis=1).clip(0)**2
    knn_prob=(weights[:,:,None]*base["neighbor_targets"][indices]).sum(axis=1)/(weights.sum(axis=1,keepdims=True)+1e-9)
    return (1-base["knn_weight"])*lr_prob+base["knn_weight"]*knn_prob


def fit_ani_validation_model():
    raw=np.vstack([train[FEATURES].to_numpy(),valid[FEATURES].to_numpy(),test[FEATURES].to_numpy()])
    scaler=StandardScaler().fit(raw);scaled=scaler.transform(raw)
    # Match Ani's original feature-only transductive preprocessing exactly.
    pca=PCA(30,random_state=0).fit(scaled)
    with warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        gmm=GaussianMixture(30,covariance_type="diag",n_init=3,random_state=0).fit(pca.transform(scaled))
    assert gmm.converged_
    Xl=scaled[:len(train)][known];Yl=original_y[known]
    Xu=np.vstack([scaled[:len(train)][~known],scaled[len(train)+len(valid):]])
    unlabeled_ids=np.r_[train.ID.to_numpy()[~known],test.ID.to_numpy()]
    assert len(unlabeled_ids)==1326 and len(set(unlabeled_ids))==1326
    assert set(unlabeled_ids).isdisjoint(valid.ID)
    XL=Xl;YL=Yl;round_audits=[]
    for round_number in range(1,3):
        base=fit_ani_base(XL,YL)
        labeled_responsibility=gmm.predict_proba(pca.transform(XL))
        normalized_targets=YL/YL.sum(axis=1,keepdims=True)
        cluster_rates=(labeled_responsibility.T@normalized_targets+normalized_targets.mean(axis=0))/(labeled_responsibility.sum(axis=0)[:,None]+1)
        gmm_prob=gmm.predict_proba(pca.transform(Xu))@cluster_rates
        combined=0.8*predict_ani_base(base,Xu)+0.2*gmm_prob
        keep=combined.max(axis=1)>=0.8
        generated=threshold_predictions(combined,0.3)[0]
        XL=np.vstack([Xl,Xu[keep]]);YL=np.vstack([Yl,generated[keep]])
        round_audits.append({"round":round_number,"generated_trials":int(keep.sum()),"verified_training_trials":157})
    final_base=fit_ani_base(XL,YL)
    packet={"kind":"ani_validation_stage","scaler":scaler,"pca":pca,"gmm":gmm,"base":final_base,
            "source_commit":ANI_COMMIT,"features":FEATURES,"reasons":REASONS,"native_cutoff":0.3,
            "verified_training_trials":157,"validation_labels_in_training":False,
            "unsupervised_fit_rows":1561,"self_training_rounds":2,"generated_trial_ids":unlabeled_ids[keep],
            "final_training_scaled_X":XL,"final_training_targets":YL,"round_audits":round_audits}
    audit=pd.DataFrame({"ID":unlabeled_ids,"accepted_in_final_round":keep})
    for j,r in enumerate(REASONS):audit[r]=np.where(keep,generated[:,j],-100)
    audit.to_csv(OUT/"datasets/ani_pseudo_labels.csv",index=False)
    pd.DataFrame(round_audits).to_csv(OUT/"ani_round_audit.csv",index=False)
    return packet


def predict_ani(model,raw):return predict_ani_base(model["base"],model["scaler"].transform(np.asarray(raw)))

with threadpool_limits(limits=1):ani=fit_ani_validation_model()
ani_probability=predict_ani(ani,valid[FEATURES].to_numpy())
ours_probability=ours_probabilities(ours,valid[FEATURES].to_numpy())
ours_prediction,_=threshold_predictions(ours_probability,0.5)
ani_prediction,_=threshold_predictions(ani_probability,0.3)
np.testing.assert_allclose(f1_score(truth,ours_prediction,average="samples",zero_division=0),0.9102564102564102,atol=1e-14)
np.testing.assert_allclose(f1_score(truth,ani_prediction,average="samples",zero_division=0),0.8931623931623932,atol=1e-14)
np.testing.assert_allclose(f1_score(truth,ani_prediction,average="macro",zero_division=0),0.7137002637312235,atol=1e-14)
assert len(ani["generated_trial_ids"])==985
joblib.dump(ours,OUT/"models/ours_component.joblib",compress=3)
joblib.dump(ani,OUT/"models/ani_validation_component.joblib",compress=3)
print("Ani's native validation scores and 985 pseudo-labeled trials reproduced; no validation-label refit.")


Ani's native validation scores and 985 pseudo-labeled trials reproduced; no validation-label refit.


In [3]:
def trial_f1(prediction):
    return 2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))


def reason_strings(prediction):return [" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]

ours_trial=trial_f1(ours_prediction);ani_trial=trial_f1(ani_prediction)
complement=valid[["ID"]].copy()
complement["actual"]=reason_strings(truth);complement["ours"]=reason_strings(ours_prediction);complement["ani"]=reason_strings(ani_prediction)
complement["ours_trial_f1"]=ours_trial;complement["ani_trial_f1"]=ani_trial
complement["ani_better"]=ani_trial>ours_trial+1e-12;complement["ours_better"]=ours_trial>ani_trial+1e-12
complement["different_reason_set"]=(ours_prediction!=ani_prediction).any(axis=1)
complement.to_csv(OUT/"complementary_errors.csv",index=False)
complement_summary={"different_reason_sets":int(complement.different_reason_set.sum()),"ani_better_trials":int(complement.ani_better.sum()),
                    "ours_better_trials":int(complement.ours_better.sum()),"equal_trial_f1":int((abs(ani_trial-ours_trial)<=1e-12).sum())}
print("Complementary errors:",complement_summary)


def aligned_ensemble_scores(ours_prob,ani_prob,alpha):
    assert 0<=alpha<=1
    own=logit(np.clip(ours_prob,EPSILON,1-EPSILON))-logit(0.5)
    other=logit(np.clip(ani_prob,EPSILON,1-EPSILON))-logit(0.3)
    evidence=(1-alpha)*own+alpha*other
    return expit(evidence),evidence

ROWS=[];SCORE_CACHE={}
for alpha in ANI_WEIGHTS:
    name=f"ani_weight{alpha:g}".replace(".","p")
    scores,evidence=aligned_ensemble_scores(ours_probability,ani_probability,alpha)
    prediction,empty=threshold_predictions(scores,0.5)
    if alpha==0:np.testing.assert_array_equal(prediction,ours_prediction)
    if alpha==1:np.testing.assert_array_equal(prediction,ani_prediction)
    metrics={"sample_f1":f1_score(truth,prediction,average="samples",zero_division=0),
             "macro_f1":f1_score(truth,prediction,average="macro",zero_division=0),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0)))}
    ROWS.append({"experiment":name,"ani_weight":alpha,"ours_weight":1-alpha,**metrics,
                 "delta_vs_ours":metrics["sample_f1"]-ours_trial.mean(),"fallback_trials":int(empty.sum()),
                 "changed_trials_vs_ours":int((prediction!=ours_prediction).any(axis=1).sum()),
                 "improved_trials_vs_ours":int((trial_f1(prediction)>ours_trial+1e-12).sum()),
                 "worsened_trials_vs_ours":int((trial_f1(prediction)<ours_trial-1e-12).sum()),"kaggle_score":np.nan})
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        frame[f"ours_probability_{r}"]=ours_probability[:,j];frame[f"ani_probability_{r}"]=ani_probability[:,j]
        frame[f"ensemble_decision_score_{r}"]=scores[:,j];frame[f"centered_log_odds_{r}"]=evidence[:,j]
        frame[f"predicted_{r}"]=prediction[:,j]
    frame["used_fallback"]=empty;frame["trial_f1"]=trial_f1(prediction)
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    policy={"ani_weight":alpha,"ours_weight":1-alpha,"method":"threshold-centered log-odds pooling",
            "ours_native_cutoff":0.5,"ani_native_cutoff":0.3,"ensemble_score_cutoff":0.5,"fallback":True,"clip_epsilon":EPSILON,
            "components":{"ours":"models/ours_component.joblib","ani":"models/ani_validation_component.joblib"},
            "score_interpretation":"decision score, not a calibrated marginal probability"}
    (OUT/"models"/f"{name}_policy.json").write_text(json.dumps(policy,indent=2)+"\n")
    SCORE_CACHE[name]=scores
comparison=pd.DataFrame(ROWS)
comparison.to_csv(OUT/"comparison.csv",index=False)
ranked=comparison.sort_values(["sample_f1","ani_weight"],ascending=[False,True],kind="stable")
best=ranked.iloc[0]
print("Best ensemble:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(comparison[["ani_weight","sample_f1","macro_f1","delta_vs_ours","changed_trials_vs_ours","improved_trials_vs_ours","worsened_trials_vs_ours"]].round(6))


Complementary errors: {'different_reason_sets': 10, 'ani_better_trials': 2, 'ours_better_trials': 5, 'equal_trial_f1': 71}
Best ensemble: ani_weight0 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,ani_weight,sample_f1,macro_f1,delta_vs_ours,changed_trials_vs_ours,improved_trials_vs_ours,worsened_trials_vs_ours
0,0.00,0.910256,0.811421,0.000000,0,0,0
1,0.10,0.897436,0.801106,-0.012821,1,0,1
2,0.25,0.897436,0.801106,-0.012821,1,0,1
3,0.50,0.888889,0.776906,-0.021368,3,0,3
4,1.00,0.893162,0.713700,-0.017094,10,2,5


In [4]:
loaded_ours=joblib.load(OUT/"models/ours_component.joblib")
loaded_ani=joblib.load(OUT/"models/ani_validation_component.joblib")
p_ours=ours_probabilities(loaded_ours,valid[FEATURES].to_numpy());p_ani=predict_ani(loaded_ani,valid[FEATURES].to_numpy())
np.testing.assert_allclose(p_ours,ours_probability,rtol=1e-12,atol=1e-12)
np.testing.assert_allclose(p_ani,ani_probability,rtol=1e-12,atol=1e-12)
for row in comparison.itertuples(index=False):
    policy=json.loads((OUT/"models"/f"{row.experiment}_policy.json").read_text())
    score,_=aligned_ensemble_scores(p_ours,p_ani,policy["ani_weight"])
    prediction,_=threshold_predictions(score,0.5)
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    np.testing.assert_array_equal(prediction,frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int))
    np.testing.assert_allclose(score,frame[[f"ensemble_decision_score_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-12)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
# Independent final Ani-base refit from persisted augmented inputs/targets.
with threadpool_limits(limits=1):refitted_base=fit_ani_base(loaded_ani["final_training_scaled_X"],loaded_ani["final_training_targets"])
np.testing.assert_allclose(predict_ani_base(refitted_base,loaded_ani["scaler"].transform(valid[FEATURES].to_numpy())),p_ani,rtol=1e-8,atol=1e-10)
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("Native endpoints, source reproduction, model reloads, all policies/scores, final-base refit, and protected artifacts verified.")


Native endpoints, source reproduction, model reloads, all policies/scores, final-base refit, and protected artifacts verified.


In [5]:
candidate=None
if best.sample_f1>ours_trial.mean()+1e-12:
    p_ours_test=ours_probabilities(loaded_ours,test[FEATURES].to_numpy())
    p_ani_test=predict_ani(loaded_ani,test[FEATURES].to_numpy())
    scores,_=aligned_ensemble_scores(p_ours_test,p_ani_test,float(best.ani_weight))
    prediction,empty=threshold_predictions(scores,0.5)
    sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert sample.ID.is_unique and set(sample.ID)==set(test.ID)
    output=pd.DataFrame({"ID":test.ID,"Predicted":reason_strings(prediction)})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_ensemble_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "policy_file":f"models/{best.experiment}_policy.json","test_rows":312,"test_fallback_trials":int(empty.sum()),
               "kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"ani_source_commit":ANI_COMMIT,"ani_source_notebook_sha256":ANI_SNAPSHOT_SHA256,
          "ani_weights":ANI_WEIGHTS,"configurations":5,"mixed_ensembles":3,"verified_training_labels_each_component":157,
          "validation_labels_added_to_training":False,"ani_unsupervised_preprocessing_fit_rows":1561,
          "ani_transductive_features_used":True,"our_preprocessing_changed":False,
          "method":"threshold-centered log-odds pooling","ours_native_cutoff":0.5,"ani_native_cutoff":0.3,
          "ensemble_decision_score_cutoff":0.5,"fallback":True,"clip_epsilon":EPSILON,
          "reference_sample_f1":float(ours_trial.mean()),"reference_macro_f1":float(f1_score(truth,ours_prediction,average="macro",zero_division=0)),
          "ani_native_sample_f1":float(ani_trial.mean()),"ani_native_macro_f1":float(f1_score(truth,ani_prediction,average="macro",zero_division=0)),
          "complementary_errors":complement_summary,"best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),
          "best_macro_f1":float(best.macro_f1),"leader_changed":bool(best.sample_f1>ours_trial.mean()+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,
          "component_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"models").glob('*.joblib'))},
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Current model and Ani ensemble","",
       f"Source Ani commit: `{ANI_COMMIT}`, notebook snapshot saved in ani_source/. "
       "Ani's validation-stage scores and 985 pseudo-labeled trials are reproduced, with only 157 verified training labels. "
       "Its original feature-only transductive scaler/PCA/GMM use train+validation+test embeddings; no validation or test "
       "labels enter supervised training. No final 235-label refit was performed. Our DBSCAN classifier stays unchanged.","",
       "## Decision-score alignment","",
       "Our five independent scores use cutoff 0.5; Ani's multinomial/neighbor scores use cutoff 0.3. "
       "Center each model's log-odds at its own cutoff, then blend those evidence scores by Ani weight alpha. "
       "An ensemble score sigmoid(evidence)>=0.5 selects a reason, with highest-score fallback when empty. "
       "This preserves native decisions at alpha=0 and alpha=1. The ensemble scores are decision scores, not calibrated "
       "marginal probabilities; no raw probability average or extra cutoff optimization was used.","",
       "## Results","",
       "| Ani weight | Sample F1 | Macro F1 | Change vs ours | Changed trials | Improved trials | Worsened trials |",
       "|---:|---:|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.ani_weight:g} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_ours:+.6f} | {row.changed_trials_vs_ours} | {row.improved_trials_vs_ours} | {row.worsened_trials_vs_ours} |")
lines.extend(["",f"Complementary errors: {complement_summary}. Best setting: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Verification and saved artifacts","",
       "Both native endpoint controls reproduce their validation scores and labels. Saved component models and policy "
       "files recreate all ensemble scores and label sets, independently rescoring to the recorded metrics. "
       "The final Ani base is independently refitted from its saved augmented inputs/targets with matching probabilities. "
       "Protected original models and CSVs retain their hashes.","",
       "models/ saves both native component models and every decision policy. The Ani component also preserves the "
       "scaler/PCA/GMM, final augmented fitting inputs/targets, and pseudo-label trial IDs. datasets/ani_pseudo_labels.csv "
       "and ani_round_audit.csv document its self-training. complementary_errors.csv records trial-level differences; "
       "validation_predictions/ saves component probabilities, centered log-odds, ensemble decision scores, and labels. "
       "comparison.csv and settings.json preserve the grid, scores, source checksums, and software versions.","",
       "This is exploratory model selection on the same 78 validation trials. No independent Kaggle improvement is "
       "established. The ensemble does not use the new self-training model; the two ideas are evaluated separately. "
       "No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No mixture improves sample F1; retain our existing DBSCAN model.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best ensemble:",best.experiment,"sample F1:",best.sample_f1,"Leader changed:",settings['leader_changed'])
print("Candidate:",candidate)


Best ensemble: ani_weight0 sample F1: 0.9102564102564102 Leader changed: False
Candidate: None
